# P02 Weather EDA

Exploratory analysis of hourly weather for six cities, read from `P02_WEATHER.MART`.

Runs unchanged in three places:

- **Snowflake Notebook**: uses the active session; add the `altair` package in the notebook's Packages menu.
- **Local Jupyter or VS Code**: `pip install snowflake-snowpark-python pandas altair`, set `SNOW_CONNECTION` to a connection in `~/.snowflake/config.toml`.
- **Deepnote**: same as local; store the connection settings as Deepnote secrets, never in the notebook.

Order of work: data quality first, then patterns, then anomalies, then write findings at the end.

In [ ]:
import os

import altair as alt
import pandas as pd

try:
    from snowflake.snowpark.context import get_active_session

    session = get_active_session()
    IN_SNOWFLAKE = True
except Exception:
    from snowflake.snowpark import Session

    session = Session.builder.config("connection_name", os.environ.get("SNOW_CONNECTION", "p02")).create()
    IN_SNOWFLAKE = False

session.sql("USE DATABASE P02_WEATHER").collect()
session.sql("USE WAREHOUSE P02_WH").collect()


def q(sql: str) -> pd.DataFrame:
    """Run SQL in Snowflake and return a pandas DataFrame (column names are upper case)."""
    return session.sql(sql).to_pandas()

## 1. Data quality first

Expected: 8760 hours per city per full year, 8784 in leap years, fewer in the current year. Anything else is a question before any chart.

In [ ]:
coverage = q("""
    SELECT CITY_SLUG, YEAR(TS_UTC) AS YEAR, COUNT(*) AS HOURS,
           COUNT_IF(TEMPERATURE_C IS NULL) AS NULL_TEMPERATURE
    FROM STG.WEATHER_HOURLY
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
coverage.pivot(index="YEAR", columns="CITY_SLUG", values="HOURS")

## 2. Seasonal cycle

Southern hemisphere cities should peak in December to February. If they do not, the time zone or the hemisphere label is wrong.

In [ ]:
monthly = q("""
    SELECT CITY_NAME, MONTH(DAY_LOCAL) AS MONTH, AVG(TEMP_MEAN_C) AS TEMP_C
    FROM MART.V_DAILY
    WHERE HOURS = 24
    GROUP BY 1, 2
""")
alt.Chart(monthly).mark_line(point=True).encode(
    x=alt.X("MONTH:O", title="Month"),
    y=alt.Y("TEMP_C:Q", title="Mean temperature, C"),
    color=alt.Color("CITY_NAME:N", title="City"),
).properties(title="Seasonal cycle", width=640, height=320)

## 3. Daily cycle in local time

The data is stored in UTC. Converting to each city's time zone is what makes the afternoon peak line up.

In [ ]:
diurnal = q("""
    SELECT CITY_NAME, HOUR(TS_LOCAL) AS LOCAL_HOUR, AVG(TEMPERATURE_C) AS TEMP_C
    FROM MART.V_HOURLY_LOCAL
    GROUP BY 1, 2
""")
alt.Chart(diurnal).mark_line().encode(
    x=alt.X("LOCAL_HOUR:O", title="Local hour"),
    y=alt.Y("TEMP_C:Q", title="Mean temperature, C"),
    color="CITY_NAME:N",
).properties(title="Daily cycle (local time)", width=640, height=320)

## 4. Anomalies

An anomaly is the difference between a day's mean temperature and the city's climatology for that day of year. The z-score scales it by the usual spread for that day.

In [ ]:
extremes = q("""
    SELECT CITY_NAME, DAY_LOCAL, ROUND(TEMP_MEAN_C, 1) AS TEMP_C,
           ROUND(ANOMALY_C, 1) AS ANOMALY_C, ROUND(Z_SCORE, 2) AS Z_SCORE
    FROM MART.V_DAILY_ANOMALY
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CITY_NAME ORDER BY ABS(Z_SCORE) DESC) <= 3
    ORDER BY CITY_NAME, ABS(Z_SCORE) DESC
""")
extremes

In [ ]:
yearly_anomaly = q("""
    SELECT CITY_NAME, YEAR(DAY_LOCAL) AS YEAR, AVG(ANOMALY_C) AS MEAN_ANOMALY_C, COUNT(*) AS DAYS
    FROM MART.V_DAILY_ANOMALY
    GROUP BY 1, 2
    HAVING COUNT(*) >= 300
""")
alt.Chart(yearly_anomaly).mark_line(point=True).encode(
    x=alt.X("YEAR:O", title="Year"),
    y=alt.Y("MEAN_ANOMALY_C:Q", title="Mean anomaly, C"),
    color="CITY_NAME:N",
).properties(title="Yearly mean anomaly vs climatology (full years only)", width=640, height=320)

## 5. Precipitation

Heavy rain day: at least 20 mm in a local day.

In [ ]:
heavy = q("""
    SELECT CITY_NAME, YEAR(MONTH_LOCAL) AS YEAR, SUM(HEAVY_RAIN_DAYS) AS HEAVY_RAIN_DAYS, SUM(PRECIP_MM) AS PRECIP_MM
    FROM MART.V_MONTHLY
    GROUP BY 1, 2
""")
alt.Chart(heavy).mark_bar().encode(
    x=alt.X("YEAR:O", title="Year"),
    y=alt.Y("HEAVY_RAIN_DAYS:Q", title="Days with at least 20 mm"),
    color="CITY_NAME:N",
    column=alt.Column("CITY_NAME:N", title=None),
).properties(width=110, height=220)

## 6. Findings

Write three to five findings, each with the chart or table that supports it and one caveat.

1. ...
2. ...
3. ...

Open questions for a next iteration:

- Is a ten-year climatology long enough to call a day anomalous?
- Which findings change if you use daily maximum instead of daily mean?
- What would you need to compare cities fairly at different elevations?

In [ ]:
# Local runs: close the session so the warehouse can suspend. In Snowflake Notebooks use End session instead.
if not IN_SNOWFLAKE:
    session.close()